In [8]:
# ======= Install Dependencies (only if running outside Kaggle) =======
!pip install torch torchvision rasterio joblib scikit-learn opencv-python albumentations

In [9]:

# ======= Cell 1: UNet Inference =======
import os
import csv
import torch
import numpy as np
import rasterio
from glob import glob
import torch.nn as nn

# UNet Definition
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 64); self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128); self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(128, 256); self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(256, 512); self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, 2); self.conv4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2); self.conv3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2); self.conv2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2); self.conv1 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_channels, 1)
    def forward(self, x):
        d1, p1 = self.down1(x), self.pool1(self.down1(x))
        d2, p2 = self.down2(p1), self.pool2(self.down2(p1))
        d3, p3 = self.down3(p2), self.pool3(self.down3(p2))
        d4, p4 = self.down4(p3), self.pool4(self.down4(p3))
        bn = self.bottleneck(p4)
        u4 = self.up4(bn); c4 = self.conv4(torch.cat([u4, d4], dim=1))
        u3 = self.up3(c4); c3 = self.conv3(torch.cat([u3, d3], dim=1))
        u2 = self.up2(c3); c2 = self.conv2(torch.cat([u2, d2], dim=1))
        u1 = self.up1(c2); c1 = self.conv1(torch.cat([u1, d1], dim=1))
        return torch.sigmoid(self.outc(c1))

# RLE Encode
import numpy as np

def rle_encode(mask):
    if np.sum(mask) == 0:
        return " "
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    runs[::2] -= 1
    return " ".join(map(str, runs))

# Inference
from torch.profiler import profile, ProfilerActivity

TEST_DIR = '/kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data'
UNET_WEIGHT_PATH = '/kaggle/input/weights/unet.pkl'
OUT_CSV = '/kaggle/working/3_unet.csv'
LOG_FILE = '/kaggle/working/unet_model_logs.txt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = UNet(in_channels=4, out_channels=1).to(DEVICE)
state = torch.load(UNET_WEIGHT_PATH, map_location=DEVICE)
model.load_state_dict(state)
model.eval()

# Params and Ops
num_params = sum(p.numel() for p in model.parameters())
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof_op:
    dummy = torch.randn(1, 4, 256, 256).to(DEVICE)
    _ = model(dummy)
    prof_op.step()
ops = sum(e.count for e in prof_op.key_averages())

with open(LOG_FILE, 'w') as f:
    f.write(f"Logs for UNet model\nParameters: {num_params}\nOperations: {ops}\n")

with open(OUT_CSV, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['id', 'segmentation'])
    for path in sorted(glob(os.path.join(TEST_DIR, '*.tif'))):
        img_id = os.path.splitext(os.path.basename(path))[0]
        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)/255.0
        inp = torch.from_numpy(img).unsqueeze(0).to(DEVICE)
        out = model(inp)
        mask = (out>0.5).cpu().numpy().squeeze().astype(np.uint8)
        writer.writerow([img_id, rle_encode(mask)])

print(f"UNet: CSV at {OUT_CSV}, logs at {LOG_FILE}")




/tmp/ipykernel_31/973302334.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(UNET_WEIGHT_PATH, map_location=DEVICE)


UNet: CSV at /kaggle/working/3_unet.csv, logs at /kaggle/working/unet_model_logs.txt


In [11]:
# ======= Cell 2: Random Forest Inference =======
import os
import csv
import joblib
import numpy as np
import rasterio
from glob import glob

def rle_encode(mask):
    if np.sum(mask) == 0:
        return " "
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    runs[::2] -= 1
    return " ".join(map(str, runs))
    
# Paths (Kaggle input/output)
TEST_DIR = '/kaggle/input/cloud-masking-test-set-satellite-cmp25-course/test/test/data'
RF_WEIGHT_PATH = '/kaggle/input/weights/random_forest.pkl'
OUT_CSV_RF = '/kaggle/working/3_rf.csv'
LOG_FILE_RF = '/kaggle/working/rf_model_logs.txt'

# Load Random Forest
rf = joblib.load(RF_WEIGHT_PATH)

# Count params and ops (approx)
num_params = sum(t.tree_.node_count for t in rf.estimators_)
ops = num_params

# Write logs
with open(LOG_FILE_RF, 'w') as f:
    f.write(f"Logs for Random Forest model\nParameters (nodes): {num_params}\nOperations (node checks): {ops}\n")

# Run inference and write CSV
with open(OUT_CSV_RF, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['id', 'segmentation'])
    for path in sorted(glob(os.path.join(TEST_DIR, '*.tif'))):
        img_id = os.path.splitext(os.path.basename(path))[0]
        with rasterio.open(path) as src:
            img = src.read([1,2,3,4]).astype(np.float32)
        features = img.reshape(4, -1).T
        mask = rf.predict(features).reshape(src.height, src.width)
        writer.writerow([img_id, rle_encode(mask)])

print(f"RF results saved to {OUT_CSV_RF}, logs to {LOG_FILE_RF}")


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  25 out of  25 | elapsed:    0.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  25 out of  25 | elapsed:    0.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  25 out of  25 | elapsed:    0.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  25 out of  25 | elapsed:    0.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  25 out of  25 | elapsed:    0.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  25 out of  25 | elapsed:    0.5s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]

RF results saved to /kaggle/working/3_rf.csv, logs to /kaggle/working/rf_model_logs.txt


[Parallel(n_jobs=4)]: Done  25 out of  25 | elapsed:    0.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  25 out of  25 | elapsed:    0.2s finished
